# 5G 用户预测

二分类任务，预测用户是否为 5G 用户。正样本极少（约 1.3%），评价指标为 AUC。
本 Notebook 包含：数据探索、划分与预处理，以及逻辑回归、随机森林和 LightGBM 的训练与对比。

## 环境与依赖

In [ ]:
import os, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.base import clone
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (roc_auc_score, roc_curve, precision_recall_curve,
                             average_precision_score)
from lightgbm import LGBMClassifier

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 110

SEED = 42
FIG_DIR = 'output/figures'
os.makedirs(FIG_DIR, exist_ok=True)

## 一、数据加载与概览

In [ ]:
df = pd.read_csv('train.csv')
print('数据规模:', df.shape)
df.head()

In [ ]:
categorical_cols = [col for col in df.columns if col.startswith('cat_')]
numerical_cols = [col for col in df.columns if col.startswith('num_')]
print('离散特征数:', len(categorical_cols), ' 连续特征数:', len(numerical_cols))
print('正样本占比: {:.4f}'.format(df['target'].mean()))
print('缺失值总数:', int(df.isnull().sum().sum()))
print('\n各离散特征取值数:')
print(df[categorical_cols].nunique().sort_values(ascending=False))

**数据观察：**
1. 无缺失值，无需插补。
2. 5G 用户占比仅 1.32%，存在严重类别不平衡，需进行类别加权。
3. `cat_12` 取值数高达 11 万，实为高基数特征，直接独热编码会导致维度爆炸，这里当作连续特征处理。

## 二、探索性数据分析（EDA）

In [ ]:
# 目标变量分布
target_counts = df['target'].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(['非5G (0)', '5G (1)'], target_counts.values, color=['#6c8ebf', '#b85450'])
for index, count in enumerate(target_counts.values):
    percentage = count / len(df) * 100
    ax.text(index, count, f'{count}\n{percentage:.2f}%', ha='center', va='bottom')
ax.set_title('目标变量分布（类别极度不平衡）')
ax.set_ylabel('样本数')
plt.tight_layout(); plt.savefig(f'{FIG_DIR}/01_target_dist.png'); plt.show()

In [ ]:
# 数值特征与 target 的相关性
features_to_correlate = numerical_cols + ['target']
correlation_matrix = df[features_to_correlate].corr()
target_correlation = correlation_matrix['target'].drop('target')

# 降序排列相关系数绝对值并取 Top15
sorted_correlation = target_correlation.abs().sort_values(ascending=False)
top_15_features = target_correlation.loc[sorted_correlation.index].head(15)

fig, ax = plt.subplots(figsize=(6, 5))
ax.barh(top_15_features.index[::-1], top_15_features.values[::-1], color='#6c8ebf')
ax.set_title('数值特征与 target 的相关性 Top15')
ax.set_xlabel('Pearson 相关系数')
plt.tight_layout(); plt.savefig(f'{FIG_DIR}/02_num_corr.png'); plt.show()

相关系数整体偏低（最高仅 0.12 左右），说明特征与标签之间线性关系很弱。非线性模型（树模型）应有明显优势。

In [ ]:
# 重要特征在不同类别下的分布
important_features = ['num_37', 'num_3', 'num_10', 'num_30']
fig, axes = plt.subplots(2, 2, figsize=(11, 7))
for ax, feature_name in zip(axes.flatten(), important_features):
    for class_label, color in [(0, '#6c8ebf'), (1, '#b85450')]:
        class_subset = df[df['target'] == class_label]
        ax.hist(class_subset[feature_name], bins=50, alpha=0.5,
                color=color, label=f'target={class_label}', density=True)
    ax.set_title(feature_name)
    ax.legend()
    ax.set_yscale('log')
plt.suptitle('重要数值特征在不同类别下的分布')
plt.tight_layout(); plt.savefig(f'{FIG_DIR}/03_feat_dist.png'); plt.show()

## 三、数据划分与预处理

分层切分出 20% 作为测试集，剩余 80% 用于 5 折交叉验证。
不同模型的预处理策略：
- **逻辑回归**：离散特征做 OneHotEncoder，数值特征做 StandardScaler。
- **随机森林**：离散特征做 OrdinalEncoder，数值特征 passthrough。
- **LightGBM**：离散特征直接转为 category 类型。

In [ ]:
y = df['target'].astype(int).values
X = df.drop(columns=['id', 'target'])
sample_ids = df['id'].values
categorical_for_encoding = [col for col in categorical_cols if col != 'cat_12']

X_train, X_test, y_train, y_test, ids_train, ids_test = train_test_split(
    X, y, sample_ids, test_size=0.2, stratify=y, random_state=SEED)
print(f'train {len(X_train)}, test {len(X_test)}（test 正样本 {int(y_test.sum())}）')

### 评估函数：5 折 CV + 测试集评估

In [ ]:
def cross_validate_auc(model, X, y, n_splits=5):
    stratified_kfold = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    auc_scores = []
    for train_indices, val_indices in stratified_kfold.split(X, y):
        cloned_model = clone(model)
        cloned_model.fit(X.iloc[train_indices], y[train_indices])
        val_predictions = cloned_model.predict_proba(X.iloc[val_indices])[:, 1]
        auc_scores.append(roc_auc_score(y[val_indices], val_predictions))
    return np.array(auc_scores)


def evaluate_model(model_name, model, X_train, y_train, X_test, y_test):
    start_time = time.time()
    cv_auc_scores = cross_validate_auc(model, X_train, y_train)
    final_model = clone(model)
    final_model.fit(X_train, y_train)
    test_probabilities = final_model.predict_proba(X_test)[:, 1]
    test_auc = roc_auc_score(y_test, test_probabilities)
    average_precision = average_precision_score(y_test, test_probabilities)
    elapsed_time = time.time() - start_time
    print(f'[{model_name}] CV AUC = {cv_auc_scores.mean():.4f} ± {cv_auc_scores.std():.4f} | '
          f'test AUC = {test_auc:.4f} | AP = {average_precision:.4f} | {elapsed_time:.0f}s')
    return {
        'model_name': model_name,
        'cv_scores': cv_auc_scores,
        'test_auc': test_auc,
        'test_predictions': test_probabilities,
        'average_precision': average_precision,
        'final_model': final_model
    }


evaluation_results = []

## 四、模型训练

### 1. 逻辑回归（Baseline）

使用 `class_weight='balanced'` 处理类别不平衡。

In [ ]:
lr_preprocessor = ColumnTransformer([
    ('onehot', OneHotEncoder(handle_unknown='ignore'), categorical_for_encoding),
    ('scaler', StandardScaler(), numerical_cols + ['cat_12']),
])
lr_pipeline = Pipeline([
    ('preprocessor', lr_preprocessor),
    ('classifier', LogisticRegression(C=0.5, max_iter=2000, class_weight='balanced', n_jobs=-1))
])
evaluation_results.append(evaluate_model('LogisticRegression', lr_pipeline, X_train, y_train, X_test, y_test))

### 2. 随机森林

使用 `class_weight='balanced'`。

In [ ]:
rf_preprocessor = ColumnTransformer([
    ('ordinal', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), categorical_for_encoding),
    ('numeric', 'passthrough', numerical_cols + ['cat_12']),
])
rf_pipeline = Pipeline([
    ('preprocessor', rf_preprocessor),
    ('classifier', RandomForestClassifier(
        n_estimators=120, max_depth=25, min_samples_leaf=10,
        class_weight='balanced', n_jobs=-1, random_state=SEED
    ))
])
evaluation_results.append(evaluate_model('RandomForest', rf_pipeline, X_train, y_train, X_test, y_test))

### 3. LightGBM

使用 `scale_pos_weight` 处理类别不平衡。

In [ ]:
X_train_lgb = X_train.copy()
X_test_lgb = X_test.copy()
for col in categorical_for_encoding:
    X_train_lgb[col] = X_train_lgb[col].astype('category')
    X_test_lgb[col] = X_test_lgb[col].astype('category')
positive_weight = (y_train == 0).sum() / (y_train == 1).sum()

lgbm_classifier = LGBMClassifier(
    n_estimators=400, learning_rate=0.05, num_leaves=63,
    subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=positive_weight, random_state=SEED,
    n_jobs=-1, verbose=-1
)
evaluation_results.append(evaluate_model('LightGBM', lgbm_classifier, X_train_lgb, y_train, X_test_lgb, y_test))

## 五、模型对比

In [ ]:
model_names = [res['model_name'] for res in evaluation_results]
cv_means = [res['cv_scores'].mean() for res in evaluation_results]
cv_stds = [res['cv_scores'].std() for res in evaluation_results]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].bar(model_names, cv_means, yerr=cv_stds,
            color=['#8c564b', '#2ca02c', '#1f77b4'], capsize=5)
axes[0].set_ylim(0.5, 1.0); axes[0].set_ylabel('AUC')
axes[0].set_title('5 折交叉验证 AUC')

for res in evaluation_results:
    fpr, tpr, _ = roc_curve(y_test, res['test_predictions'])
    axes[1].plot(fpr, tpr, label=f"{res['model_name']} ({res['test_auc']:.4f})")
axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.4)
axes[1].set_title('测试集 ROC 曲线')
axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR'); axes[1].legend()
plt.tight_layout(); plt.savefig(f'{FIG_DIR}/04_model_compare.png'); plt.show()

In [ ]:
# PR 曲线
fig, ax = plt.subplots(figsize=(6, 5))
for res in evaluation_results:
    precision, recall, _ = precision_recall_curve(y_test, res['test_predictions'])
    ax.plot(recall, precision, label=f"{res['model_name']} (AP={res['average_precision']:.4f})")
ax.set_title('测试集 PR 曲线')
ax.set_xlabel('Recall'); ax.set_ylabel('Precision'); ax.legend()
plt.tight_layout(); plt.savefig(f'{FIG_DIR}/05_pr_curve.png'); plt.show()

In [ ]:
summary_df = pd.DataFrame({
    'model': model_names,
    'cv_auc_mean': np.round(cv_means, 4),
    'cv_auc_std': np.round(cv_stds, 4),
    'test_auc': np.round([res['test_auc'] for res in evaluation_results], 4),
    'test_AP': np.round([res['average_precision'] for res in evaluation_results], 4),
})
summary_df.to_csv('output/cv_results.csv', index=False)
summary_df

## 六、特征重要性

以最优秀的 LightGBM 模型分裂次数来衡量特征重要性。

In [ ]:
best_lgb_model = evaluation_results[-1]['final_model']
feature_importances = pd.Series(
    best_lgb_model.feature_importances_,
    index=best_lgb_model.feature_name_
)
top_20_importances = feature_importances.sort_values(ascending=False).head(20)

fig, ax = plt.subplots(figsize=(6, 6))
ax.barh(top_20_importances.index[::-1], top_20_importances.values[::-1], color='#1f77b4')
ax.set_title('LightGBM 特征重要性 Top20'); ax.set_xlabel('分裂次数')
plt.tight_layout(); plt.savefig(f'{FIG_DIR}/06_lgb_importance.png'); plt.show()

## 七、保存预测结果

选择测试集 AUC 最高的模型生成预测概率文件并保存。

In [ ]:
best_model_result = max(evaluation_results, key=lambda res: res['test_auc'])
submission_df = pd.DataFrame({
    'id': ids_test,
    'target': best_model_result['test_predictions']
})
submission_df.to_csv('output/submission.csv', index=False)
print(f"最优模型: {best_model_result['model_name']}，test AUC = {best_model_result['test_auc']:.4f}")
submission_df.head()

## 结论
1. 测试集 AUC 表现：LightGBM 与随机森林表现基本一致，显著优于逻辑回归，说明非线性特征交互重要。
2. 类别极度不平衡情况下，PR 曲线与 AP 比 ROC 指标更能反映对正类的识别能力。
3. 后续优化方向：细化超参搜索（如早停与贝叶斯优化）、目标编码 `cat_12`、模型集成融合（Stacking）。